# 11 - Overlay: ticker mentions vs price (AUTO, weekly)

Auto-picks the most-mentioned tickers in the window and overlays mentions vs
price. `FREQ='W'` = weekly (smoother); set `'D'` for daily or `'M'` for monthly.
The price line is forward-filled so it has no weekend gaps.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# window comes from update_data.py (same toggle as live vs backtest)
from update_data import START_DATE, END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError('prices.parquet not found - run  python pull_bloomberg_prices.py  first.')
    px = pd.read_parquet(PRICES_PATH); px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    # daily close, then made CONTINUOUS (forward-fill weekends/holidays) so the
    # line is smooth with no gaps. Clip to the window.
    one = prices[prices['symbol'] == symbol].sort_values('date')
    s = one.set_index('date')['px_last']
    if not s.empty:
        s = s.asfreq('D').ffill()
    return clip_series(s)


In [ ]:
HOW_MANY = 6      # how many top tickers to plot
FREQ = 'W'        # 'W' weekly (smooth), 'D' daily, 'M' monthly
ROLL = 7          # daily smoothing (only used when FREQ='D')

In [ ]:
counts = pd.read_parquet(os.path.join(P, 'daily_ticker_counts.parquet'))
counts['date'] = pd.to_datetime(counts['date']); counts = clip_dates(counts, 'date')
tickers = counts.groupby('ticker')['mention_count'].sum().sort_values(ascending=False).head(HOW_MANY).index.tolist()
print('auto tickers:', tickers)
prices = load_prices()
for ticker in tickers:
    m = counts[counts['ticker'] == ticker].sort_values('date').set_index('date')['mention_count'].asfreq('D').fillna(0)
    px = price_series(prices, ticker)
    if px.empty:
        print('skip', ticker, '- no price rows'); continue
    if FREQ == 'D':
        if ROLL > 1: m = m.rolling(ROLL).mean()
    else:
        m = m.resample(FREQ).sum(); px = px.resample(FREQ).last()
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.plot(m.index, m.values, color='tab:blue', linewidth=1.8, label='mentions')
    ax1.set_ylabel(f'mentions per {FREQ}', color='tab:blue'); ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax2 = ax1.twinx(); ax2.plot(px.index, px.values, color='tab:red', linewidth=1.8, label='price')
    ax2.set_ylabel('price (USD)', color='tab:red'); ax2.tick_params(axis='y', labelcolor='tab:red')
    ax1.set_title(f'{ticker}: mentions vs price ({FREQ})'); ax1.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()